# Codificar categorías

La mayoría de los modelos calculan con números, así que una columna de categorías tiene que convertirse en una o más columnas numéricas antes de que puedan usarla. La codificación no es neutral: los números le dicen al modelo qué son las categorías, si tienen un orden y cuánto distan entre sí. Una codificación incorrecta le dice algo falso, y algunos modelos se lo creen.

## Una tabla pequeña

El combustible y el estado de seis coches: una categoría sin orden y otra con orden ([Tipos de variables](deep-dives/kinds_of_features.ipynb)):

In [ ]:
import pandas as pd

cars = pd.DataFrame({
    "fuel": ["diesel", "petrol", "diesel", "electric", "petrol", "hybrid"],
    "condition": ["good", "excellent", "fair", "excellent", "good", "poor"],
})
cars

,fuel,condition
0,diesel,good
1,petrol,excellent
2,diesel,fair
3,electric,excellent
4,petrol,good
5,hybrid,poor


## Codificación ordinal

La codificación más sencilla da a cada categoría un número entero. Lo hace el `OrdinalEncoder` de scikit-learn, que aprende las categorías con `fit`. Si no se le indica otra cosa, las numera por orden alfabético:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

OrdinalEncoder().set_output(transform="pandas").fit_transform(cars).assign(
    fuel_text=cars["fuel"], condition_text=cars["condition"])

,fuel,condition,fuel_text,condition_text
0,0.0,2.0,diesel,good
1,3.0,0.0,petrol,excellent
2,0.0,1.0,diesel,fair
3,1.0,0.0,electric,excellent
4,3.0,2.0,petrol,good
5,2.0,3.0,hybrid,poor


Para el estado, el orden alfabético no tiene sentido: "excellent" es 0 y "poor" es 3. El orden hay que darlo explícitamente, de peor a mejor:

In [ ]:
condition_encoder = OrdinalEncoder(categories=[["poor", "fair", "good", "excellent"]]).set_output(transform="pandas")
condition_encoder.fit_transform(cars[["condition"]]).assign(condition_text=cars["condition"])

,condition,condition_text
0,2.0,good
1,3.0,excellent
2,1.0,fair
3,3.0,excellent
4,2.0,good
5,0.0,poor


Ahora 0 es poor y 3 es excellent. La codificación le dice dos cosas al modelo: que los estados están ordenados, lo cual es cierto, y que los saltos entre ellos son iguales (de poor a fair hay tanto salto como de good a excellent), lo cual es una suposición.

Para el combustible, cualquier orden es falso. Codificado como diesel 0, electric 1, hybrid 2, petrol 3, un modelo lineal leería electric como a medio camino entre diesel y hybrid, y petrol como tres veces más lejos de diesel que electric. Una categoría sin orden necesita otra codificación, al menos para los modelos que tratan los números como cantidades.

scikit-learn también tiene un `LabelEncoder`. A pesar de su nombre, no es para las variables: codifica el **objetivo** (*target*) de un clasificador, una sola columna, siempre en orden alfabético, sin ninguna de las opciones de `OrdinalEncoder`. En un pipeline de preprocesamiento, usa `OrdinalEncoder`.

## Codificación *one-hot*

La **codificación *one-hot*** convierte una columna de categorías en una columna por categoría, con un 1 en la columna de la categoría de la fila y un 0 en las demás: en cada fila solo una columna está «encendida» (*hot*). A las columnas nuevas a veces se las llama variables *dummy*:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

fuel_encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
fuel_encoder.fit_transform(cars[["fuel"]]).assign(fuel=cars["fuel"])

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol,fuel
0,1.0,0.0,0.0,0.0,diesel
1,0.0,0.0,0.0,1.0,petrol
2,1.0,0.0,0.0,0.0,diesel
3,0.0,1.0,0.0,0.0,electric
4,0.0,0.0,0.0,1.0,petrol
5,0.0,0.0,1.0,0.0,hybrid


Sin orden y sin distancias: todos los pares de combustibles son igual de distintos. El coste es una columna por categoría.

Por defecto, `OneHotEncoder` devuelve una matriz *dispersa* (*sparse*), que solo guarda las posiciones de los unos. Con muchas categorías casi todos los valores son 0, y una matriz dispersa ahorra la mayor parte de la memoria; `sparse_output=False` da una tabla normal, más fácil de mirar.

### Por qué no `pandas.get_dummies`

El `get_dummies` de pandas hace lo mismo en una sola llamada, pero no aprende nada: crea una columna por cada categoría presente *en los datos que recibe*. Unos datos nuevos con menos combustibles reciben menos columnas:

In [ ]:
new_cars = pd.DataFrame({"fuel": ["petrol", "diesel"]})
pd.get_dummies(new_cars["fuel"])

,diesel,petrol
0,False,True
1,True,False


Dos columnas en lugar de cuatro, y un modelo entrenado con cuatro columnas no puede usarlas. `OneHotEncoder` recuerda las categorías que vio en `fit`, así que los datos nuevos siempre reciben las mismas columnas:

In [ ]:
fuel_encoder.transform(new_cars)

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol
0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0


### Categorías que solo aparecen en datos nuevos

Un coche que funciona con gas licuado del petróleo, un combustible que el codificador nunca vio. Por defecto, `OneHotEncoder` lo rechaza:

In [ ]:
unknown_fuel = pd.DataFrame({"fuel": ["lpg"]})
try:
    fuel_encoder.transform(unknown_fuel)
except ValueError as error:
    print(error)

Found unknown categories ['lpg'] in column 0 during transform


Un error es un buen comportamiento por defecto: dice que ha aparecido algo nuevo. Cuando se esperan categorías nuevas, `handle_unknown="ignore"` las codifica como una fila de ceros, «ninguna de las categorías conocidas»:

In [ ]:
tolerant_fuel_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore").set_output(transform="pandas")
tolerant_fuel_encoder.fit(cars[["fuel"]]).transform(unknown_fuel)

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol
0,0.0,0.0,0.0,0.0


### Eliminar una columna

Las columnas *one-hot* de una categoría siempre suman 1, así que cualquiera de ellas puede calcularse a partir de las demás. Para una regresión lineal sin regularización, esto repite la información del término independiente, y los pesos dejan de tener una solución única (la «trampa de las variables *dummy*»). `drop="first"` elimina una columna por variable, y `drop="if_binary"` solo en las categorías con dos valores. Los árboles y los modelos regularizados (como la regresión logística de scikit-learn y la mayoría de los modelos que se usan en la práctica) no lo necesitan ([Regularización](../supervised-learning/ml-fundamentals/regularization.ipynb)). Ten en cuenta que, con una columna eliminada, una categoría desconocida y la eliminada se codifican ambas como ceros: se vuelven indistinguibles.

## Una tabla real: los ingresos en el censo de 1994

El dataset *Adult*: 48 842 personas del censo de Estados Unidos de 1994, con su edad, estudios, ocupación, situación familiar y país de origen. El objetivo es si sus ingresos superaban los 50 000 dólares al año. Lo publica OpenML; scikit-learn lo descarga la primera vez:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml("adult", version=2, as_frame=True, data_home="data").frame
X = adult.drop(columns="class")
y = (adult["class"] == ">50K").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
categorical = X.select_dtypes("category").columns.to_list()
X_train[categorical].nunique()

workclass          8
education         16
marital-status     7
occupation        14
relationship       6
race               5
sex                2
native-country    41
dtype: int64

Ocho variables categóricas, con entre 2 y 41 categorías. Dos de ellas muestran los dos casos difíciles: una categoría ordenada y una categoría con muchos valores.

### Una categoría ordenada, ya codificada

Los estudios, frente a la columna `education-num` que viene con ellos:

In [ ]:
X_train.groupby("education", observed=True)["education-num"].agg(["min", "max", "size"]).sort_values("min")

,min,max,size
education,,,
Preschool,1,1,69
1st-4th,2,2,204
5th-6th,3,3,399
7th-8th,4,4,756
9th,5,5,602
10th,6,6,1118
11th,7,7,1450
12th,8,8,527
HS-grad,9,9,12689


Cada nivel de estudios tiene exactamente un número, desde preescolar (1) hasta doctorado (16), en el orden de los niveles: `education-num` es la codificación ordinal de `education`, con el orden dado a mano. Un modelo debería recibir una de las dos, no ambas, ya que son la misma información. Como número, supone saltos iguales: pasar de 9.º a 10.º curso cuenta tanto como pasar de un grado a un máster. Si eso es dudoso, codificar los estudios con *one-hot* elimina la suposición, a costa de eliminar también el orden.

### Una categoría con muchos valores

In [ ]:
country_counts = X_train["native-country"].value_counts()
country_counts.head(3), country_counts.tail(3)

(native-country
 United-States    35101
 Mexico             763
 Philippines        243
 Name: count, dtype: int64,
 native-country
 Laos                  17
 Hungary               16
 Holand-Netherlands     1
 Name: count, dtype: int64)

De 39 073 filas de entrenamiento, unas 35 000 son de Estados Unidos. Muchos países tienen unas pocas decenas de filas, y Países Bajos tiene una. La codificación *one-hot* da a cada país una columna, la mayoría casi siempre a 0, y un modelo no puede aprender nada fiable sobre un país a partir de una fila, ni de diecisiete.

`OneHotEncoder` puede agrupar las categorías raras en una columna *infrequent*, ya sea por debajo de un número mínimo de filas (`min_frequency`) o más allá de un número máximo de columnas (`max_categories`). `handle_unknown="infrequent_if_exist"` envía las categorías nunca vistas en el entrenamiento a esa misma columna, que es su sitio natural:

In [ ]:
country_encoder = OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False)
country_encoder.fit(X_train[["native-country"]])
len(country_encoder.categories_[0]), len(country_encoder.get_feature_names_out())

(42, 24)

42 categorías (los 41 países y el valor ausente, que el codificador trata como una categoría más) se convierten en 24 columnas: una por cada uno de los 22 países con al menos 50 filas de entrenamiento, una para el valor ausente y una para todos los demás.

### Codificación por objetivo *(profundización)*

La **codificación por objetivo** (*target encoding*) sustituye cada categoría por la media del objetivo en las filas de entrenamiento de esa categoría: aquí, la proporción de personas de cada país que ganan más de 50 000 dólares. Una sola columna, haya las categorías que haya, y un número que significa algo para la tarea.

La media de una categoría con pocas filas es poco fiable, así que se acerca a la media de todas las filas, tanto más cuantas menos filas tenga la categoría. Con `smooth=20`, una categoría con $n$ filas recibe $(n \cdot \text{su media} + 20 \cdot \text{la media global}) / (n + 20)$:

In [ ]:
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # los pliegues del ajuste cruzado, fijos para que los resultados se repitan

country_target_encoder = TargetEncoder(smooth=20, cv=folds).fit(X_train[["native-country"]], y_train)
encoded = pd.Series(country_target_encoder.encodings_[0], index=country_target_encoder.categories_[0])
y_train.groupby(X_train["native-country"], observed=True).agg(rows="size", share_above_50k="mean").assign(
    encoded=encoded).sort_values("rows").head(6).round(3)

,rows,share_above_50k,encoded
native-country,,,
Holand-Netherlands,1,0.000,0.228
Hungary,16,0.375,0.300
Laos,17,0.118,0.183
Honduras,17,0.059,0.156
Cambodia,19,0.368,0.302
Scotland,19,0.158,0.200


In [ ]:
y_train.mean().round(3)

np.float64(0.239)

La única persona de Países Bajos ganaba menos de 50 000 dólares, una media de 0; la codificación da 0,228, cerca del 0,239 global, porque una fila no dice casi nada. Los países con unas pocas decenas de filas se acercan solo en parte. El valor por defecto, `smooth="auto"`, estima a partir de los datos cuánto acercar; no acerca nada una categoría cuyas filas tienen todas el mismo objetivo, como le pasa siempre a una categoría de una sola fila, así que un valor fijo puede ser más seguro cuando hay muchas categorías diminutas.

La codificación por objetivo usa el objetivo, y eso la hace peligrosa. Una fila codificada con una media que incluye su propio objetivo lleva un trozo de la respuesta: con una categoría de una fila y sin suavizado, la codificación *es* la respuesta. Para evitarlo, `TargetEncoder.fit_transform` sobre las filas de entrenamiento usa **ajuste cruzado** (*cross-fitting*): divide las filas en pliegues y codifica cada fila con las medias de los otros pliegues. Por eso `fit_transform` da valores distintos de `fit` seguido de `transform`, que codifica cada fila con medias que la incluyen:

In [ ]:
import numpy as np

cross_fitted = TargetEncoder(smooth=20, cv=folds).fit_transform(X_train[["native-country"]], y_train)
not_cross_fitted = TargetEncoder(smooth=20, cv=folds).fit(X_train[["native-country"]], y_train).transform(X_train[["native-country"]])
np.allclose(cross_fitted, not_cross_fitted)

False

Dentro de un pipeline esto se maneja correctamente: `Pipeline.fit` llama a `fit_transform`, y la predicción llama a `transform` con las codificaciones aprendidas de todas las filas de entrenamiento. Llamar a mano a `fit` y luego a `transform` sobre las filas de entrenamiento es el error. [Dónde termina la preparación de datos](where_preparation_ends.ipynb) muestra lo grande que es la fuga que provoca.

## Qué codificación, para qué modelo

Tres codificaciones de las siete variables categóricas sin orden (los estudios se dejan fuera, ya que `education-num` ya los codifica), cada una con un modelo lineal, una regresión logística, y con gradient boosting, un modelo hecho de árboles. Cada pipeline rellena las categorías ausentes con la más frecuente y escala las variables numéricas; validación cruzada de 5 pliegues sobre las filas de entrenamiento:

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numerical = X.select_dtypes("number").columns.to_list()
nominal = [column for column in categorical if column != "education"]
encoders = {
    "one-hot": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "target": TargetEncoder(cv=folds),
}
models = {
    "logistic regression": LogisticRegression(max_iter=2000),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}


def with_encoder(encoder, model):
    preprocessing = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"), encoder), nominal),
        (StandardScaler(), numerical),
    )
    return make_pipeline(preprocessing, model)


pd.DataFrame({
    model_name: {
        encoder_name: cross_val_score(with_encoder(encoder, model), X_train, y_train, cv=5).mean()
        for encoder_name, encoder in encoders.items()
    }
    for model_name, model in models.items()
}).round(3)

,logistic regression,gradient boosting
one-hot,0.851,0.873
ordinal,0.824,0.873
target,0.847,0.872


Las diferencias pequeñas significan poco: solo con los pliegues ya se mueve un poco cada exactitud.

- **La regresión logística pierde casi tres puntos con códigos ordinales.** Multiplica cada variable por un peso, así que lee los códigos como cantidades: la ocupación 7 cuenta como siete veces la ocupación 1. La codificación *one-hot* da a cada categoría su propio peso.
- **Al gradient boosting le da igual.** Un árbol divide una variable por umbrales (código ≤ 3, código > 3), y con varias divisiones puede aislar cualquier grupo de códigos, así que el orden falso no le cuesta casi nada. La codificación ordinal es una opción habitual y barata para los árboles.
- **La codificación por objetivo**, que sustituye cada categoría por el objetivo medio de sus filas de entrenamiento (la sección de profundización de arriba), queda cerca de *one-hot* en ambos, con una sola columna por variable.

| Tipo de categoría | Modelos lineales, k-NN, SVM, redes | Árboles y gradient boosting |
| --- | --- | --- |
| Sin orden, pocas categorías | *One-hot* | *One-hot* u ordinal |
| Sin orden, muchas categorías | *One-hot* agrupando las raras; codificación por objetivo | Ordinal, codificación por objetivo o soporte nativo |
| Con orden | Ordinal con el orden dado, o *one-hot* si los saltos son claramente desiguales | Ordinal con el orden dado |

Las bibliotecas de gradient boosting también pueden recibir las categorías directamente, sin ninguna codificación: el `HistGradientBoostingClassifier` de scikit-learn, que por defecto lee como categorías las columnas del tipo `category` de pandas, y XGBoost con `enable_categorical=True` ([XGBoost](../supervised-learning/algorithms/tree-based/xgboost.ipynb)). Las redes neuronales a menudo aprenden en su lugar un ***embedding*** para cada categoría: un vector corto de números, aprendido durante el entrenamiento, que coloca cerca las categorías parecidas (Redes neuronales *(próximamente)*).